<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/day3/notebooks/2_bert_new_data_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Day 3: Your Turn · BERT on New Data · Solutions

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every ✏️ exercise is filled in with one possible answer and
a short comment. Your numbers depend on which dataset you picked, and will differ a little from
run to run, so read the comments as "what to look for", not as exact results.

In the BERT notebook we classified bill titles in three ways. Bill titles are an easy case,
though: they're short, formal, and full of telltale words like *Medicare* or *Internal Revenue
Code*. Most research data isn't like that.

In this notebook you do the same on data that's harder in different ways. Each table works on
one of three datasets from the Hugging Face Hub, runs the same approaches as on the bills, and
reports back to the room. At the end we compare across tables: where does BERT help most, and
where does nothing really work?

| | Dataset | The texts | The task | Why it's interesting |
|---|---|---|---|---|
| 🌱 `claims` | **Environmental claims** | 2,647 sentences from companies' annual reports, sustainability reports and earnings calls | Does the sentence make an environmental claim? *yes* or *no* | Greenwashing. The line between making a claim and describing something is subtle. |
| 🌡️ `climate` | **Climate sentiment** | 1,320 climate-related paragraphs from corporate disclosures | Is the paragraph about climate *risk*, *opportunity*, or neither (*neutral*)? | How firms talk about climate change. Paragraphs instead of sentences, and the smallest of the three datasets. |
| 💊 `health` | **PUBHEALTH** | 12,258 public health claims checked by fact-checkers and news sites | Is the claim *true*, *false*, a *mixture*, or *unproven*? | Health misinformation. Can a model tell whether a claim is true from its wording alone? |

### How this notebook is organized

| Step | | Rough time |
|---|---|---|
| 1. Pick your dataset and place your bets | core | 3 min |
| 2. Load and look | core | 7 min |
| 3. Be the annotator | core | 5 min |
| 4. Baselines | core | 5 min |
| 5. Zero-shot | core | 8 min |
| 6. Fine-tuning | core | 12 min |
| 7. Results, and reporting back | core | 5 min |
| 8. Deep dives | deep dive | open-ended |

Work through the core steps as a table. If you finish early, pick a deep dive.

**A few practical tips**

- Most of the code is the same as in the BERT notebook, packed into fewer cells. If something
  looks unfamiliar, the matching section there explains it step by step.
- Run the cells **in order**. If Colab resets your session after a break, use
  *Runtime → Run before* to catch up.

> ⚠️ **Turn on the GPU before you start.** *Runtime → Change runtime type → T4 GPU.*

## 0. Setup

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers

from datasets import load_dataset, Dataset
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

# Fix the random seed of Python, NumPy and PyTorch in one go, so results are repeatable
SEED = 42
set_seed(SEED)

pd.set_option("display.max_colwidth", 200)

# Use the GPU if there is one
device = "cuda" if torch.cuda.is_available() else "cpu"

print("transformers:", transformers.__version__)
print("Running on:  ", device.upper())
if device == "cuda":
    print("GPU:         ", torch.cuda.get_device_name(0))
else:
    print("\n⚠️  No GPU found. Runtime → Change runtime type → T4 GPU, then run this cell again.")

---
## 1. Pick your dataset and place your bets

Set `DATASET` to the dataset your table is working on: `"claims"`, `"climate"` or `"health"`.

In [ ]:
DATASET = "claims"   # 🌱 "claims", 🌡️ "climate" or 💊 "health"

### Place your bets

Before you look at any data: how well will each approach do on your dataset? Think back to
the scores on the bill titles. Will this task be easier or harder? Which approach will gain
or lose the most? Agree on a guess as a table and write it down as a **macro-F1** between 0
and 1.

(Macro-F1 computes the F1 score for each label separately and then averages them, so a rare
label counts as much as a common one. A perfect classifier scores 1.)

We'll come back to your bets at the end.

In [ ]:
our_bets = {
    "TF-IDF + logistic regression": None,   # replace None with your guess, e.g. 0.70
    "Zero-shot (BART-MNLI)": None,
    "Fine-tuned DistilBERT": None,
}

---
## 2. Load and look

### 2.1 Settings for each dataset

Everything that differs between the three datasets is collected in one place: where the data
lives on the Hub, which column holds the text, how many tokens of each text we keep, and how
we describe the labels to the zero-shot model in Section 5.

In [ ]:
CONFIGS = {
    "claims": {
        "name": "🌱 Environmental claims",
        "title": "Environmental claims",       # for plot titles (no emoji)
        "hub_id": "climatebert/environmental_claims",
        "subset": None,
        "text_column": "text",
        "max_length": 128,
        "zs_template": "This sentence is {}.",
        "zs_labels": {                         # what the zero-shot model reads → the dataset's label
            "an environmental claim by a company": "yes",
            "about something else": "no",
        },
    },
    "climate": {
        "name": "🌡️ Climate sentiment",
        "title": "Climate sentiment",
        "hub_id": "climatebert/climate_sentiment",
        "subset": None,
        "text_column": "text",
        "max_length": 256,
        "zs_template": "This text is about {}.",
        "zs_labels": {
            "climate-related risks": "risk",
            "climate-related opportunities": "opportunity",
            "neutral facts": "neutral",
        },
    },
    "health": {
        "name": "💊 PUBHEALTH",
        "title": "PUBHEALTH",
        "hub_id": "Jezzarax/pubhealth-converted",
        "subset": "pubhealth_source",
        "text_column": "claim",
        "max_length": 128,
        "label_order": ["true", "false", "mixture", "unproven"],
        "zs_template": "This claim is {}.",
        "zs_labels": {
            "true": "true",
            "false": "false",
            "partly true": "mixture",
            "unproven": "unproven",
        },
    },
}

cfg = CONFIGS[DATASET]
MAX_LENGTH = cfg["max_length"]
print("Working on:", cfg["name"])

### 2.2 Loading from the Hub

In the BERT notebook we downloaded single Parquet files of the bills by hand, because that
dataset is huge. These datasets are small, so we use the standard route: `load_dataset` with
the dataset's name on the Hub. It downloads the data, caches it, and returns a `DatasetDict`
with one entry per split.

> **A note on PUBHEALTH.** The official copy on the Hub (`ImperialCollegeLondon/health_fact`)
> is an older dataset that loads through a Python script, and current versions of `datasets`
> no longer run such scripts. We use a copy that someone converted to plain Parquet files
> (`Jezzarax/pubhealth-converted`). Its conversion report lists exactly which rows were dropped:
> 65 malformed rows out of 12,323. Checking that kind of thing is part of using community
> uploads.

In [ ]:
ds = load_dataset(cfg["hub_id"], cfg["subset"])
ds

Each split is a `Dataset`, a table with typed columns. In the two ClimateBERT datasets the
`label` column is a `ClassLabel`: it stores integers and keeps the label names alongside. In
PUBHEALTH the labels are plain strings. The next cell turns every split into a pandas DataFrame
with the same three columns, whichever dataset you picked: `text`, `label_name`, and an integer
`label`.

In [ ]:
label_feature = ds["train"].features["label"]
if hasattr(label_feature, "names"):       # a ClassLabel: integers, with the names stored alongside
    labels = list(label_feature.names)
else:                                     # plain strings, as in PUBHEALTH
    labels = cfg["label_order"]
label2id = {name: i for i, name in enumerate(labels)}
id2label = {i: name for name, i in label2id.items()}


def to_frame(split):
    df = ds[split].to_pandas().rename(columns={cfg["text_column"]: "text"})
    df["label_name"] = df["label"].map(id2label) if hasattr(label_feature, "names") else df["label"]
    df["label"] = df["label_name"].map(label2id)
    df["text"] = df["text"].fillna("").str.strip()
    df = df[(df["text"] != "") & df["label"].notna()]
    return df.astype({"label": int}).reset_index(drop=True)


print("Label IDs:", label2id)

### 2.3 Training, validation and test

We need three splits, as in the BERT notebook. Environmental claims and PUBHEALTH come with all
three. Climate sentiment only has a training and a test split, so we set aside 15% of its
training data for validation.

Two more steps keep the comparison honest and quick:

- **No text in two places.** If the exact same text appears in both the training and the test
  data, part of the test is a memory test. We remove duplicate texts within the training data,
  and texts in validation or test that already appeared in an earlier split.
- **At most 600 texts for validation and for test.** Only PUBHEALTH is bigger than that. A
  stratified sample of 600 keeps zero-shot classification reasonably fast.

In [ ]:
train_df = to_frame("train")
if "validation" in ds:
    val_df = to_frame("validation")
else:
    train_df, val_df = train_test_split(train_df, test_size=0.15, stratify=train_df["label"],
                                        random_state=SEED)
test_df = to_frame("test")

# No duplicates within training, and nothing in validation or test that appeared before
n_before = len(train_df) + len(val_df) + len(test_df)
train_df = train_df.drop_duplicates(subset="text")
seen = set(train_df["text"])
val_df = val_df.drop_duplicates(subset="text")
val_df = val_df[~val_df["text"].isin(seen)]
seen |= set(val_df["text"])
test_df = test_df.drop_duplicates(subset="text")
test_df = test_df[~test_df["text"].isin(seen)]
print(f"Removed {n_before - len(train_df) - len(val_df) - len(test_df)} duplicate texts")

MAX_EVAL = 600

def cap(df, n=MAX_EVAL):
    if len(df) <= n:
        return df.reset_index(drop=True)
    sample, _ = train_test_split(df, train_size=n, stratify=df["label"], random_state=SEED)
    return sample.reset_index(drop=True)

train_df = train_df.reset_index(drop=True)
val_df, test_df = cap(val_df), cap(test_df)
y_train, y_val, y_test = train_df["label"].values, val_df["label"].values, test_df["label"].values
print(f"train: {len(train_df)}   validation: {len(val_df)}   test: {len(test_df)}")

### 2.4 Look before you model

In [ ]:
pd.DataFrame({
    "train": train_df["label_name"].value_counts(normalize=True),
    "validation": val_df["label_name"].value_counts(normalize=True),
    "test": test_df["label_name"].value_counts(normalize=True),
}).reindex(labels).round(3)

Are the labels balanced? If one label is rare, keep an eye on it. Models tend to neglect rare
labels, which accuracy hides and macro-F1 shows.

Three examples of each label:

In [ ]:
for name in labels:
    print(f"── {name} ──")
    for t in train_df.loc[train_df["label_name"] == name, "text"].sample(3, random_state=SEED):
        print("  •", t[:300] + ("..." if len(t) > 300 else ""))
    print()

And how long are the texts, in DistilBERT tokens?

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

n_tokens = train_df["text"].apply(lambda t: len(tokenizer(t)["input_ids"]))
print(n_tokens.describe().round(1))
print(f"\nTexts that fit in MAX_LENGTH = {MAX_LENGTH} tokens: {(n_tokens <= MAX_LENGTH).mean():.1%}")

plt.figure(figsize=(8, 3))
plt.hist(n_tokens, bins=50, color="#34B233", edgecolor="white")
plt.axvline(MAX_LENGTH, color="black", linestyle="--", label=f"MAX_LENGTH = {MAX_LENGTH}")
plt.xlabel("Tokens per text")
plt.ylabel("Number of texts")
plt.legend()
plt.show()

Texts longer than `MAX_LENGTH` are cut off (truncated) when we fine-tune. (A warning about a
sequence being "longer than the specified maximum" in the cell above is harmless: we only
counted tokens there.) If a large share of your texts doesn't fit, you could raise `MAX_LENGTH`
in `CONFIGS`, up to 512, at the cost of slower training.

### 2.5 Read the dataset card

Before you trust a dataset's labels, find out where they came from. Open your dataset's page
on the Hub:

- 🌱 [climatebert/environmental_claims](https://huggingface.co/datasets/climatebert/environmental_claims)
- 🌡️ [climatebert/climate_sentiment](https://huggingface.co/datasets/climatebert/climate_sentiment)
- 💊 [Jezzarax/pubhealth-converted](https://huggingface.co/datasets/Jezzarax/pubhealth-converted),
  and the original, [ImperialCollegeLondon/health_fact](https://huggingface.co/datasets/ImperialCollegeLondon/health_fact)

Discuss as a table:

1. Where do the texts come from?
2. Who assigned the labels, and how? Does the card say how much the annotators agreed?
3. Under which license may you use the data? In a paper? In a commercial product?
4. What does the card *not* tell you that you'd want to know?

> **✅ What the cards say**
>
> - 🌱 **Environmental claims**: sentences from annual reports, sustainability reports and
>   earnings call transcripts, mainly of large listed companies. Labeled by the authors and by
>   students at the University of Zurich with majors in finance and sustainable finance. The
>   card points to the paper's appendix for the guidelines and doesn't report agreement.
>   License: CC BY-NC-SA 4.0, so research and teaching are fine, commercial use is not.
> - 🌡️ **Climate sentiment**: paragraphs from annual and sustainability reports, labeled by the
>   authors and students at the universities of Zurich and Erlangen-Nürnberg. Same license.
> - 💊 **PUBHEALTH**: claims from fact-checking sites and health news sites, with the verdicts
>   those sites gave, mapped onto four labels by the dataset's authors (Kotonya & Toni, 2020).
>   MIT license according to the original loader. The converted copy is a community upload, so
>   its conversion report is what lets you check it against the original.
>
> What's missing: the ClimateBERT cards don't report inter-annotator agreement, and they leave
> their sections on biases and limitations empty ("More Information Needed"). For a paper, you'd
> go to the original publications for those details.

---
## 3. Be the annotator

Before any model touches the data, try the task yourself. Here are ten texts from the training
data, without their labels.

In [ ]:
quiz = train_df.sample(10, random_state=SEED).reset_index(drop=True)
print("Possible labels:", labels, "\n")
for i, t in enumerate(quiz["text"]):
    print(f"[{i}] {t[:500]}{'...' if len(t) > 500 else ''}\n")

> **✏️ Exercise 1**
>
> As a table, decide on a label for each of the ten texts and type them into `my_labels`, in
> order, using the label names printed above. Then run the cell after it. How often do you
> agree with the official labels? Where you disagree, who do you think is right?

In [ ]:
# ✅ Solution
# There's nothing to copy here: the point is your own reading of the texts. For illustration,
# this "table" simply chose the most common label every time.
my_labels = [train_df["label_name"].value_counts().index[0]] * 10

# What to look for: count your agreements, but spend more time on the disagreements. Was the
# text genuinely ambiguous, did you read the category differently from the annotators, or would
# you need more context than one sentence or paragraph? If your table agrees with the labels on
# only 7 out of 10, then a model that agrees on 80% isn't doing badly at all.

In [ ]:
assert len(my_labels) == 10, "my_labels needs exactly ten labels"
check = quiz[["text", "label_name"]].copy()
check["our label"] = my_labels
check["agree"] = check["our label"] == check["label_name"]
print(f"We agreed with the official label on {check['agree'].sum()} of 10 texts")
check

Keep your agreement score in mind. Models are trained and tested against these labels, so the
labels set the ceiling: if careful readers often disagree with them, no model will score close
to 1, and a model that "gets it wrong" will sometimes be right.

---
## 4. Baselines

Two baselines this time. The first is the simplest classifier possible: always predict the most
common label. Any approach worth using has to beat it clearly. The second is the TF-IDF model
from Day 1 and the BERT notebook.

In [ ]:
def scores(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", labels=list(range(len(labels))),
                             zero_division=0),
    }

def report(name, y_true, y_pred):
    s = scores(y_true, y_pred)
    print(f"{name:<34} accuracy {s['accuracy']:.3f}   macro-F1 {s['macro_f1']:.3f}")

def per_label(y_true, y_pred):
    print(classification_report(y_true, y_pred, labels=list(range(len(labels))), target_names=labels,
                                digits=3, zero_division=0))


# We collect every approach's TEST predictions as we go, but only look at test scores in Section 7
test_predictions = {}

majority = DummyClassifier(strategy="most_frequent").fit(train_df[["text"]], y_train)
report("Always the most common label", y_val, majority.predict(val_df[["text"]]))
test_predictions["Always the most common label"] = majority.predict(test_df[["text"]])

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train = tfidf.fit_transform(train_df["text"])     # learn the vocabulary on TRAIN only
X_val = tfidf.transform(val_df["text"])
X_test = tfidf.transform(test_df["text"])

tfidf_clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
report("TF-IDF + logistic regression", y_val, tfidf_clf.predict(X_val))
test_predictions["TF-IDF + logistic regression"] = tfidf_clf.predict(X_test)

Look at accuracy and macro-F1 side by side. If they're far apart, the labels are imbalanced and
the model is probably neglecting a rare one. The per-label report shows which:

In [ ]:
per_label(y_val, tfidf_clf.predict(X_val))

> **✏️ Exercise 2**
>
> By default, logistic regression treats every training text as equally important, so the
> common labels dominate. Train a second TF-IDF model with
> `LogisticRegression(max_iter=1000, class_weight="balanced")`, which gives texts with rare
> labels more weight. What happens to accuracy, to macro-F1, and to the recall of the rarest
> label? Which of the two models would you rather use, and does that depend on your research
> question?

In [ ]:
# ✅ Solution
balanced_clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_train, y_train)
balanced_pred = balanced_clf.predict(X_val)

report("TF-IDF, unweighted", y_val, tfidf_clf.predict(X_val))
report("TF-IDF, class_weight='balanced'", y_val, balanced_pred)
print()
per_label(y_val, balanced_pred)

# What to look for: with balanced weights the rare labels get predicted more often, so their
# recall goes up, usually at the cost of some precision and often a bit of overall accuracy.
# Macro-F1 tends to improve, because it rewards doing reasonably on every label. Which model is
# better depends on what you'll do with the labels. If you want to estimate how common a rare
# label is, a model that hardly ever predicts it is useless, however high its accuracy.

> **✏️ Exercise 3**
>
> Which words does the TF-IDF model rely on most for each label? Print the ten features with
> the largest coefficients per label. `tfidf_clf.coef_` holds one row of coefficients per label
> and `tfidf.get_feature_names_out()` the matching words. (With only two labels there is a
> single row: positive values point to the second label, negative values to the first.)
>
> Do the words make sense? Do you see words that say more about *where a text comes from* or
> *how it's written* than about what it says?

In [ ]:
# ✅ Solution
words = tfidf.get_feature_names_out()
coefs = tfidf_clf.coef_
if coefs.shape[0] == 1:                       # two labels: one row, positive = second label
    coefs = np.vstack([-coefs[0], coefs[0]])

for i, name in enumerate(labels):
    top = np.argsort(coefs[i])[::-1][:10]
    print(f"{name:<12}", ", ".join(words[top]))

# What to look for:
# 🌱 claims: words about emissions, energy, waste or reductions for "yes". For "no", check
#    whether the list contains the spoken language of earnings calls ("think", "quarter", "we're").
#    If so, the model has partly learned "earnings call = no claim": a fact about the source.
# 🌡️ climate: risk words ("exposed", "risks", "physical") against opportunity words
#    ("opportunities", "renewable"). Neutral is usually the hardest to describe in words.
# 💊 health: look closely at "true". If its top words read like news headlines (agencies,
#    "study", "says"), the model may be learning what a headline looks like rather than what
#    is true. Deep dive 8.3 investigates.

---
## 5. Zero-shot

The same model and approach as in Section 9 of the BERT notebook. The model reads each text
together with a sentence such as *"This sentence is an environmental claim by a company."* and
judges how well the two fit together. The label descriptions for your dataset are in
`cfg["zs_labels"]`. They're a first attempt, and in Exercise 4 you'll try to do better.

Zero-shot classification is slow, because the model reads every text once per label. The cell
prints how long it took.

In [ ]:
zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)

def zero_shot_predict(texts, label_map, template, batch_size=16):
    outputs = zero_shot(list(texts), list(label_map), hypothesis_template=template,
                        batch_size=batch_size)
    return np.array([label2id[label_map[o["labels"][0]]] for o in outputs])

example = val_df["text"].iloc[0]
result = zero_shot(example, list(cfg["zs_labels"]), hypothesis_template=cfg["zs_template"])
print(example[:300], "\n")
for lab, score in zip(result["labels"], result["scores"]):
    print(f"{score:.3f}  {lab}")

In [ ]:
start = time.time()
zs_val_pred = zero_shot_predict(val_df["text"], cfg["zs_labels"], cfg["zs_template"])
print(f"Classified {len(val_df)} texts in {time.time() - start:.0f} seconds\n")
report("Zero-shot (BART-MNLI)", y_val, zs_val_pred)
print()
per_label(y_val, zs_val_pred)

> **✏️ Exercise 4**
>
> Zero-shot results depend heavily on how you describe the labels. Write two or three
> alternative versions of the label descriptions (and, if you like, of the template) and
> compare them on the **validation** set. Store the best one in `best_labels` and
> `best_template`: the next cell uses them on the test set.
>
> Why choose on the validation set and not on the test set?

In [ ]:
# ✅ Solution
VARIANTS = {
    "claims": {
        "original": (CONFIGS["claims"]["zs_labels"], CONFIGS["claims"]["zs_template"]),
        "commitments": ({"a claim about a company's environmental performance or commitments": "yes",
                         "a general business or financial statement": "no"}, "This sentence is {}."),
        "green or not": ({"environmentally friendly": "yes", "neutral": "no"},
                         "This statement by a company is {}."),
    },
    "climate": {
        "original": (CONFIGS["climate"]["zs_labels"], CONFIGS["climate"]["zs_template"]),
        "threat or chance": ({"a threat or cost caused by climate change": "risk",
                              "a business opportunity from climate action": "opportunity",
                              "a neutral description": "neutral"}, "This text describes {}."),
        "tone": ({"negative": "risk", "positive": "opportunity", "neutral": "neutral"},
                 "The tone of this text about climate change is {}."),
    },
    "health": {
        "original": (CONFIGS["health"]["zs_labels"], CONFIGS["health"]["zs_template"]),
        "accurate": ({"accurate": "true", "inaccurate": "false", "partly accurate": "mixture",
                      "not supported by evidence": "unproven"}, "This health claim is {}."),
        "news or rumor": ({"reliable news": "true", "misinformation": "false",
                           "misleading": "mixture", "an unverified rumor": "unproven"}, "This is {}."),
    },
}

results = {}
for name, (label_map, template) in VARIANTS[DATASET].items():
    pred = zero_shot_predict(val_df["text"], label_map, template)
    results[name] = scores(y_val, pred)["macro_f1"]
    print(f"{name:<18} validation macro-F1 = {results[name]:.3f}")

best = max(results, key=results.get)
best_labels, best_template = VARIANTS[DATASET][best]
print("\nBest on the validation set:", best)

# Why the validation set: every choice you make by looking at a score is a form of fitting. If
# you picked the wording with the best TEST score, the test score would no longer be an honest
# estimate of how the method does on new texts. With a few hundred validation texts, small
# differences between wordings may also just be noise.
#
# For 💊 health, expect all versions to stay low. Whether a claim is true is mostly not
# something you can read off the claim itself, so the model has little to go on.

In [ ]:
test_predictions["Zero-shot (BART-MNLI)"] = zero_shot_predict(test_df["text"], best_labels, best_template)

---
## 6. Fine-tuning

The recipe from Section 11 of the BERT notebook, packed into one function so we can reuse it in
the deep dives: tokenize the texts, load DistilBERT with a fresh classification head, train for
three epochs, and evaluate on the validation set after every epoch. The settings are the same as
for the bills.

In [ ]:
def compute_metrics(eval_pred):
    logits, true_labels = eval_pred
    return scores(true_labels, np.argmax(logits, axis=-1))


def to_dataset(df, tok):
    data = Dataset.from_pandas(df[["text", "label"]].reset_index(drop=True))
    return data.map(lambda batch: tok(batch["text"], truncation=True, max_length=MAX_LENGTH),
                    batched=True)


def run_fine_tuning(train=None, checkpoint="distilbert/distilbert-base-uncased", epochs=3,
                    learning_rate=2e-5, batch_size=16, seed=SEED):
    # Fine-tune `checkpoint` on `train` (default: all of train_df); return the trainer and tokenizer
    train = train_df if train is None else train
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint, num_labels=len(labels), id2label=id2label, label2id=label2id)

    warmup = int(0.1 * math.ceil(len(train) / batch_size) * epochs)      # 10% of all steps
    args = TrainingArguments(
        output_dir="tmp-run", num_train_epochs=epochs, learning_rate=learning_rate,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=64,
        weight_decay=0.01, warmup_steps=warmup, eval_strategy="epoch", save_strategy="no",
        logging_strategy="epoch", seed=seed, fp16=(device == "cuda"), report_to="none",
    )
    trainer = Trainer(model=model, args=args,
                      train_dataset=to_dataset(train, tok), eval_dataset=to_dataset(val_df, tok),
                      data_collator=DataCollatorWithPadding(tokenizer=tok),
                      compute_metrics=compute_metrics)
    trainer.train()
    return trainer, tok

In [ ]:
start = time.time()
ft_trainer, ft_tokenizer = run_fine_tuning()
print(f"\nTraining took {(time.time() - start) / 60:.1f} minutes")

As in the BERT notebook, a message about newly initialized weights (or a load report with
weights marked `MISSING`) is expected here: the classification head is new, and it's what we
just trained.

How did training go, epoch by epoch?

In [ ]:
history = pd.DataFrame(ft_trainer.state.log_history)
train_loss = history.dropna(subset=["loss"])[["epoch", "loss"]]
val_scores = history.dropna(subset=["eval_loss"])[["epoch", "eval_loss", "eval_accuracy", "eval_macro_f1"]]
train_loss.merge(val_scores, on="epoch").round(3)

Training loss should go down. If the validation loss starts going up while the training loss
keeps falling, the model has begun to memorize the training texts.

We store the test predictions for Section 7:

In [ ]:
ft_test_pred = ft_trainer.predict(to_dataset(test_df, ft_tokenizer)).predictions.argmax(axis=-1)
test_predictions["Fine-tuned DistilBERT"] = ft_test_pred

> **✏️ Exercise 5**
>
> Read the model's mistakes, on the **validation** set. Get predictions with
> `ft_trainer.predict(to_dataset(val_df, ft_tokenizer))`, turn them into probabilities with
> `softmax` (from `scipy.special`), and show the ten texts the model got wrong with the highest
> confidence. For each one: is the model wrong, or is the official label debatable? Pick one
> error to share with the room.

In [ ]:
# ✅ Solution
from scipy.special import softmax

val_output = ft_trainer.predict(to_dataset(val_df, ft_tokenizer))
probs = softmax(val_output.predictions.astype("float32"), axis=-1)

errors = val_df[["text", "label_name"]].copy()
errors["predicted"] = [id2label[i] for i in probs.argmax(axis=1)]
errors["confidence"] = probs.max(axis=1).round(3)
errors = errors[errors["label_name"] != errors["predicted"]].sort_values("confidence", ascending=False)

print(f"{len(errors)} errors out of {len(val_df)} validation texts")
errors.head(10)

# What to look for: confident mistakes are the interesting ones. Some will be real model errors
# (a negation, irony, a claim hidden in a long paragraph). Others will be texts where you'd side
# with the model, which says something about the labels or the category definitions rather than
# the model. If the errors cluster in one pair of labels, that's a codebook question too.

---
## 7. Results, and reporting back

Now, and only now, we look at the test set.

In [ ]:
summary = pd.DataFrame(
    [{"approach": name, **scores(y_test, pred)} for name, pred in test_predictions.items()]
).set_index("approach").round(3)
summary

In [ ]:
ax = summary["macro_f1"].plot(kind="barh", figsize=(8, 3), color="#34B233")
ax.set_xlabel("Test macro-F1")
ax.set_xlim(0, 1.1)
ax.invert_yaxis()
for i, v in enumerate(summary["macro_f1"]):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center")
plt.title(f"{cfg['title']}: same test set, four approaches")
plt.tight_layout()
plt.show()

How did your bets do?

In [ ]:
pd.DataFrame({
    "our bet": pd.Series(our_bets, dtype="float"),
    "actual macro-F1": summary["macro_f1"],
}).reindex(summary.index)

And the fine-tuned model, label by label:

In [ ]:
per_label(y_test, ft_test_pred)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, ft_test_pred, labels=list(range(len(labels))),
                                        display_labels=labels, cmap="Greens", colorbar=False, ax=ax)
ax.set_title(f"Fine-tuned DistilBERT, test set ({cfg['title']})")
plt.tight_layout()
plt.show()

### Report back

Run the next cell and share the results card with the room. Then prepare a sentence or two for
each question:

1. **Which approach won, and by how much?** Is the gap between TF-IDF and fine-tuned BERT
   bigger or smaller than on the bill titles? Why might that be?
2. **How did your bets do?** What surprised you?
3. **One error worth sharing:** a text the model got confidently wrong, and whether you think
   the model or the label is right.
4. **Would you use this classifier in a paper?** What would you check or change first?

In [ ]:
lines = [f"{cfg['name']} · {len(train_df):,} training texts · {len(test_df)} test texts"]
for name, row in summary.iterrows():
    bet = our_bets.get(name)
    bet_text = f"   (our bet: {bet:.2f})" if isinstance(bet, (int, float)) else ""
    lines.append(f"  {name:<30} macro-F1 {row['macro_f1']:.2f}{bet_text}")
print("\n".join(lines))

### What to take away

- **The baseline still earns its place.** When a few words carry the label, TF-IDF is hard to
  beat. Fine-tuning tends to gain more when the label depends on how something is said.
- **Zero-shot works when your categories can be phrased as a sentence the model understands.**
  It's a quick first look, not a measurement instrument, and its scores swing with the wording.
- **A model can only learn what's in the text.** If the label depends on facts about the world,
  as with the truth of a health claim, a classifier can only pick up indirect cues such as
  style or source. That's a measurement problem, and no model fixes it.
- **Scores are only as good as the labels.** Your own agreement with the official labels in
  Section 3 is a useful reality check on any number in this notebook.

---
## 8. Deep dives

Pick one. Deep dive 8.1 works for every dataset, 8.2 for 🌱 and 🌡️, and 8.3 for 💊.

### 8.1 How many labels do you need? *(any dataset)*

Hand-labeling is usually the expensive part of a classification project. So how much does
performance depend on the number of labeled training texts?

> **✏️ Exercise 6**
>
> For training sets of 100, 250, 500 and 1,000 texts, and the full training set, draw a
> stratified random sample of `train_df`. On each sample, train TF-IDF + logistic regression
> and fine-tune DistilBERT (`run_fine_tuning(train=sample, epochs=5)`), and record the
> validation macro-F1 of both. Plot the two curves against the number of training texts. Do
> they cross? What does the curve mean for planning an annotation project?

In [ ]:
# ✅ Solution
sizes = [n for n in [100, 250, 500, 1000] if n < len(train_df)] + [len(train_df)]
rows = []
for n in sizes:
    if n == len(train_df):
        sample = train_df
    else:
        sample, _ = train_test_split(train_df, train_size=n, stratify=train_df["label"],
                                     random_state=SEED)
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    lr = LogisticRegression(max_iter=1000).fit(vec.fit_transform(sample["text"]), sample["label"])
    tfidf_f1 = scores(y_val, lr.predict(vec.transform(val_df["text"])))["macro_f1"]
    trainer, _ = run_fine_tuning(train=sample, epochs=5)
    bert_f1 = trainer.evaluate()["eval_macro_f1"]
    rows.append({"training texts": n, "TF-IDF": tfidf_f1, "fine-tuned DistilBERT": bert_f1})

curve = pd.DataFrame(rows).set_index("training texts")
ax = curve.plot(marker="o", figsize=(7, 4), color=["#1A1A2E", "#34B233"])
ax.set_xscale("log")
ax.set_ylabel("Validation macro-F1")
ax.set_title(f"Learning curve: {cfg['title']}")
plt.show()
curve.round(3)

# What to look for: where the curves are still climbing at the right-hand end, more labels would
# probably help; where they've flattened, extra annotation buys little. With 100 texts,
# fine-tuning sees only a handful of small batches, so the points on the left are noisy. Run it
# again with another seed to get a feel for how noisy.

### 8.2 Has someone already done this? *(🌱 and 🌡️)*

The ClimateBERT team, who built both datasets, also published fine-tuned models on the Hub:
[`climatebert/environmental-claims`](https://huggingface.co/climatebert/environmental-claims) and
[`climatebert/distilroberta-base-climate-sentiment`](https://huggingface.co/climatebert/distilroberta-base-climate-sentiment).
Before you train a model yourself, it's always worth checking the Hub. But a model you download
is only as trustworthy as what you know about how it was made.

> **✏️ Exercise 7**
>
> 1. Open the model card for your dataset's model. Which base model did it start from? What
>    data was it fine-tuned on? Does the card say which **split** was used for training?
> 2. Run the model on your test set with `pipeline("text-classification", ...)`. First print
>    `.model.config.id2label`: does the model number its labels the same way as the dataset?
>    Then map its predicted label *names* to your label IDs and compute the scores.
> 3. How much would you trust that score, given what the card does and doesn't say?

In [ ]:
# ✅ Solution
HUB_MODELS = {
    "claims": "climatebert/environmental-claims",
    "climate": "climatebert/distilroberta-base-climate-sentiment",
}

if DATASET not in HUB_MODELS:
    print("This deep dive is for the two ClimateBERT datasets.")
else:
    hub_clf = pipeline("text-classification", model=HUB_MODELS[DATASET], device=device)
    print("The model's labels:  ", hub_clf.model.config.id2label)
    print("The dataset's labels:", id2label, "\n")

    out = hub_clf(list(test_df["text"]), batch_size=32, truncation=True, max_length=512)
    hub_pred = np.array([label2id[o["label"]] for o in out])      # map by NAME, not by number
    report("ClimateBERT model from the Hub", y_test, hub_pred)
    report("Our fine-tuned DistilBERT", y_test, ft_test_pred)

# What the cards say: both models were fine-tuned on these datasets, starting from
# climatebert/distilroberta-base-climate-f, a DistilRoBERTa model that the team pre-trained
# further on climate-related text. Neither card says which split was used for training, and
# neither reports evaluation results. So the model may well have seen our test texts during
# training, and a high score here doesn't tell you how it does on new data. Before using such
# a model in a paper, you'd need to find out (the paper, the authors) or test it on texts you
# know it hasn't seen.
#
# The label numbers: the climate sentiment model counts opportunity = 0, neutral = 1, risk = 2,
# while the dataset counts risk = 0, neutral = 1, opportunity = 2. Comparing numbers instead of
# names would have silently swapped risk and opportunity. Always map by name.

> **✏️ Exercise 8**
>
> A fairer test of what ClimateBERT adds: fine-tune its *base* model,
> `climatebert/distilroberta-base-climate-f`, on our own training data with
> `run_fine_tuning(checkpoint=...)`, and compare its validation macro-F1 with DistilBERT's.
> Because you trained it yourself, you know it hasn't seen the validation set. Does
> pre-training on climate texts help? Is the difference bigger than what you'd get from
> changing only the random seed?

In [ ]:
# ✅ Solution
if DATASET not in ["claims", "climate"]:
    print("This deep dive is for the two ClimateBERT datasets.")
else:
    climate_trainer, _ = run_fine_tuning(checkpoint="climatebert/distilroberta-base-climate-f")
    distilbert_f1 = ft_trainer.evaluate()["eval_macro_f1"]
    climatebert_f1 = climate_trainer.evaluate()["eval_macro_f1"]
    print(f"DistilBERT               validation macro-F1 {distilbert_f1:.3f}")
    print(f"ClimateBERT base model   validation macro-F1 {climatebert_f1:.3f}")

# To judge the difference, rerun both with seed=1 and seed=2 and look at the spread. If the gap
# between the models is smaller than the spread between seeds, you can't conclude that domain
# pre-training helps here. A domain model is also a different model in other ways (RoBERTa
# instead of BERT, cased instead of uncased), so a difference isn't only about the climate texts.

### 8.3 What's actually in the text? *(💊)*

PUBHEALTH asks a model to tell true claims from false ones. But whether a claim is true usually
depends on the world, not on its wording. So what can a text classifier pick up on?

> **✏️ Exercise 9**
>
> The `subjects` column holds the tags that the source site gave each claim. Many claims are
> tagged *Health News*, and those read like news headlines. In the training data, make a
> crosstab of the label against whether `subjects` contains "Health News"
> (`pd.crosstab(..., normalize="index")`). What do you see? What could a model learn instead of
> truth?

In [ ]:
# ✅ Solution
if DATASET != "health":
    print("This deep dive is for PUBHEALTH.")
else:
    is_news = train_df["subjects"].fillna("").str.contains("Health News")
    print(f"{is_news.mean():.1%} of the training claims are tagged 'Health News'\n")
    print(pd.crosstab(is_news.map({True: "Health News", False: "other"}), train_df["label_name"],
                      normalize="index").round(2))

# What to look for: if the Health News items are almost all labeled "true", a large part of
# the "true" class consists of news headlines, while "false" claims come mostly from
# fact-checking sites, which check rumors. A classifier can then score well by recognizing the
# style of a news headline. That's a pattern in how the dataset was put together, not a
# judgment of truth, and it would break down on texts where headlines and rumors are mixed,
# such as social media posts. The general lesson: always ask what in the text, or in how the
# data was collected, could give the label away.

> **✏️ Exercise 10**
>
> Each claim also comes with the fact-checker's `explanation`. Train the TF-IDF model on the
> explanation instead of the claim and compare the validation scores. Then print the top words
> per label, as in Exercise 3. Why is this classifier useless for research, even if it scores
> much better?

In [ ]:
# ✅ Solution
if DATASET != "health":
    print("This deep dive is for PUBHEALTH.")
else:
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    lr = LogisticRegression(max_iter=1000).fit(vec.fit_transform(train_df["explanation"].fillna("")), y_train)
    report("TF-IDF on the claim", y_val, tfidf_clf.predict(X_val))
    report("TF-IDF on the explanation", y_val, lr.predict(vec.transform(val_df["explanation"].fillna(""))))
    print()
    words = vec.get_feature_names_out()
    for i, name in enumerate(labels):
        top = np.argsort(lr.coef_[i])[::-1][:10]
        print(f"{name:<10}", ", ".join(words[top]))

# Explanations are written after the fact-check, by the people who gave the verdict, so they
# often contain the verdict or point to it (words like "false", "no evidence" or "misleading",
# or the house style of a particular fact-checking site). The classifier learns to read off the
# answer, not to judge a claim. This is label leakage: the input contains information that
# only exists because the label was assigned. It's easy to spot here, but it happens in subtler
# forms in real projects, for instance through metadata or texts edited after the event.

---
## Sources

- **Environmental claims**: Stammbach, D., Webersinke, N., Bingler, J. A., Kraus, M., &
  Leippold, M. (2023). Environmental claim detection. *Proceedings of ACL 2023 (Short Papers)*.
  Preprint: *A dataset for detecting real-world environmental claims*, arXiv:2209.00507.
- **Climate sentiment**: Bingler, J. A., Kraus, M., Leippold, M., & Webersinke, N. (2023). How
  cheap talk in climate disclosures relates to climate initiatives, corporate emissions, and
  reputation risk. SSRN working paper 3998435.
- **PUBHEALTH**: Kotonya, N., & Toni, F. (2020). Explainable automated fact-checking for public
  health claims. *Proceedings of EMNLP 2020*. arXiv:2010.09926.
- **ClimateBERT**: Webersinke, N., Kraus, M., Bingler, J. A., & Leippold, M. (2021). ClimateBERT:
  A pretrained language model for climate-related text. arXiv preprint.